# 07 — Calibrate, đánh giá cuối và thí nghiệm theo thời gian (v2)

Đồ án Máy học — Bank Marketing (Nhóm 14). Chạy theo thứ tự từ trên xuống dưới.

Giai đoạn 7b (v2): Calibrate xác suất + chọn ngưỡng theo chi phí-lợi ích.

Thiết kế theo thời gian (A1/A2, sau góp ý 2026-11-14):
- Mô hình cuối lấy từ scripts/07_time_design.py (time_selection.json).
- Chọn phương pháp calibrate (isotonic vs Platt/sigmoid) bằng Brier trên
  dự đoán out-of-fold của train60 (CalibratedClassifierCV cv=3), rồi
  calibrate mô hình đã fit trên validation (FrozenEstimator, không chạm test).
- Ngưỡng chính chọn theo chi phí-lợi ích (C=1, R=15 — cặp tham chiếu của
  nhóm, không phải chi phí thật) trên xác suất ĐÃ CALIBRATE của validation.
  Ngưỡng tối ưu F1 và ngưỡng giữ recall>=0,7 vẫn tính để tham chiếu.
- Lưu models/model.joblib (mô hình gốc, chưa calibrate),
  models/model_calibrated.joblib (mô hình dùng trong app) và
  models/threshold.json (định dạng mới, có threshold_cost_benefit).

In [ ]:
import json
import os
import sys
from pathlib import Path
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, precision_recall_curve
from sklearn.model_selection import cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
sys.path.insert(0, str(Path("..").resolve()))
from src.config import MODELS_DIR, REPORTS_DIR, SEED
from src.data import load_clean, time_split_3way
from src.evaluate import _single_thread
from src.features import build_pipeline
COST_C, COST_R = 1.0, 15.0  # cặp tham chiếu (giả định của nhóm, không phải chi phí thật)
CLASSES = {"LogReg": LogisticRegression, "Tree": DecisionTreeClassifier,
           "KNN": KNeighborsClassifier, "HGB": HistGradientBoostingClassifier,
           "LightGBM": LGBMClassifier, "RF": RandomForestClassifier}
DEFAULT_KWARGS = {
    "LogReg": {"max_iter": 3000, "class_weight": "balanced", "random_state": SEED},
    "Tree": {"class_weight": "balanced", "random_state": SEED},
    "KNN": {"n_jobs": 1},
    "HGB": {"class_weight": "balanced", "random_state": SEED},
    "LightGBM": {"class_weight": "balanced", "verbosity": -1, "n_jobs": 1, "random_state": SEED},
    "RF": {"class_weight": "balanced_subsample", "n_jobs": 1, "random_state": SEED},
}
def estimator_from_best(name, best_params):
    kwargs = {k.replace("clf__", ""): v for k, v in best_params[name].items()}
    return CLASSES[name](**{**DEFAULT_KWARGS[name], **kwargs})
def fmt(x, nd=4):
    return f"{x:.{nd}f}".replace(".", ",")
def cost_benefit_threshold(y_true, proba, C=COST_C, R=COST_R):
    """Ngưỡng tối đa hóa lợi nhuận kỳ vọng: xếp theo proba, gọi từ trên xuống.
    Trả về (ngưỡng, top_pct, capture_pct, profit_max)."""
    order = np.argsort(-np.asarray(proba))
    y_sorted = np.asarray(y_true)[order]
    p_sorted = np.asarray(proba)[order]
    profit = R * np.cumsum(y_sorted) - C * (np.arange(len(y_sorted)) + 1)
    k = int(np.argmax(profit))
    return float(p_sorted[k]), float((k + 1) / len(y_sorted)), \
        float(y_sorted[:k + 1].sum() / y_sorted.sum()), float(profit[k])

In [ ]:
df = load_clean()
X_tr, X_va, X_te, y_tr, y_va, y_te = time_split_3way(df)
sel = json.loads((REPORTS_DIR / "time_selection.json").read_text(encoding="utf-8"))
best_params = json.loads((REPORTS_DIR / "best_params_time.json").read_text(encoding="utf-8"))
final_name = sel["final_model"]
print("Mô hình cuối (thiết kế theo thời gian):", final_name, "|", sel["selection_reason"])

base = _single_thread(build_pipeline(estimator_from_best(final_name, best_params)))
base.fit(X_tr, y_tr)

## Chọn phương pháp calibrate bằng Brier OOF trên train60 ---

In [ ]:
method_scores = {}
for method in ["isotonic", "sigmoid"]:
    cal_cv = CalibratedClassifierCV(_single_thread(build_pipeline(
        estimator_from_best(final_name, best_params))), method=method, cv=3)
    oof = cross_val_predict(cal_cv, X_tr, y_tr, method="predict_proba", n_jobs=-1)[:, 1]
    method_scores[method] = float(brier_score_loss(y_tr, oof))
    print(f"  calibrate {method}: Brier OOF (train60) = {method_scores[method]:.4f}")
method = min(method_scores, key=method_scores.get)
print("Phương pháp calibrate được chọn:", method)

## Calibrate prefit trên validation (sklearn 1.9 đã bỏ cv="prefit";

In [ ]:
# cách thay thế chính thức là bọc mô hình đã fit trong FrozenEstimator) ---
cal = CalibratedClassifierCV(FrozenEstimator(base), method=method)
cal.fit(X_va, y_va)
p_va_raw = base.predict_proba(X_va)[:, 1]
p_va_cal = cal.predict_proba(X_va)[:, 1]
brier = {"brier_val_raw": round(float(brier_score_loss(y_va, p_va_raw)), 4),
         "brier_val_calibrated": round(float(brier_score_loss(y_va, p_va_cal)), 4),
         "brier_oof_train60": {m: round(s, 4) for m, s in method_scores.items()},
         "method": method}
print("Brier trên validation (lưu ý: calibrate fit trên validation nên số này "
      f"lạc quan): gốc {brier['brier_val_raw']:.4f} -> đã calibrate "
      f"{brier['brier_val_calibrated']:.4f}. So sánh trung thực ở test trong script 09.")

## Ngưỡng trên validation (xác suất đã calibrate) ---

In [ ]:
thr_cb, top_pct, cap_pct, profit = cost_benefit_threshold(y_va, p_va_cal)
prec, rec, thr_arr = precision_recall_curve(y_va, p_va_cal)
prec, rec = prec[:-1], rec[:-1]
f1 = 2 * prec * rec / (prec + rec + 1e-12)
thr_f1 = float(thr_arr[np.argmax(f1)])
ok = rec >= 0.7
thr_r70 = float(thr_arr[ok][np.argmax(prec[ok])]) if ok.any() else float(thr_arr[-1])
print(f"Ngưỡng chi phí-lợi ích (C={COST_C}, R={COST_R}): {thr_cb:.4f} "
      f"(gọi top {top_pct * 100:.1f}%, bắt {cap_pct * 100:.1f}% yes, "
      f"lợi nhuận {profit:.0f} đơn vị trên validation)")
print(f"Ngưỡng tối ưu F1 (validation): {thr_f1:.4f} | ngưỡng recall>=0,7: {thr_r70:.4f}")

## Lưu mô hình + ngưỡng + metadata calibrate ---

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(base, MODELS_DIR / "model.joblib")
joblib.dump(cal, MODELS_DIR / "model_calibrated.joblib")
(MODELS_DIR / "threshold.json").write_text(
    json.dumps({"threshold_cost_benefit": thr_cb, "threshold_f1": thr_f1,
                "threshold_recall70": thr_r70, "calibration_method": method,
                "cost_benefit_C": COST_C, "cost_benefit_R": COST_R},
               ensure_ascii=False, indent=2), encoding="utf-8")
(REPORTS_DIR / "calibration_comparison.json").write_text(
    json.dumps({"final_model": final_name, "selection_reason": sel["selection_reason"],
                "best_params": best_params.get(final_name, {}), **brier,
                "threshold_cost_benefit": round(thr_cb, 4),
                "threshold_f1_val": round(thr_f1, 4),
                "threshold_recall70_val": round(thr_r70, 4),
                "val_top_pct_at_cb_threshold": round(top_pct, 4),
                "val_capture_pct_at_cb_threshold": round(cap_pct, 4),
                "val_profit_at_cb_threshold": round(profit, 1)},
               ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã lưu models/model.joblib + models/model_calibrated.joblib + models/threshold.json")

Giai đoạn 7c (v2): Đánh giá trên test theo thời gian.

- Mô hình đã calibrate (scripts/08) + ngưỡng chi phí-lợi ích, đánh giá trên
  test20 (20% cuối theo thời gian) — trong lần chạy này test không dùng
  để tuning, calibrate hay chọn ngưỡng; tuy nhiên test đã được mở nhiều
  lần trong toàn bộ dự án (đính chính 19/11/2026, xem báo cáo mục 4.2.8).
- Bootstrap CI 95% cho ROC-AUC / PR-AUC / F1.
- Brier trước/sau calibrate TRÊN TEST (so sánh trung thực).
- Phân tích lỗi theo nhóm + theo quý trên test (B4).
- Gain/lift trên test (RQ5).
- B2: permutation importance và SHAP tính trên VALIDATION (không dùng test).
- Lưu models/model_metadata.json (ngày huấn luyện, phiên bản thư viện,
  hash dữ liệu, metric) cạnh mô hình.

In [ ]:
import hashlib
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import (ConfusionMatrixDisplay, PrecisionRecallDisplay,
                             RocCurveDisplay, average_precision_score,
                             brier_score_loss, roc_auc_score)
sys.path.insert(0, str(Path("..").resolve()))
from src.config import (FIGURES_DIR, MODELS_DIR, RAW_PATH, REPORTS_DIR, SEED)
from src.data import load_clean, time_split_3way
from src.evaluate import bootstrap_ci, compute_metrics, gain_lift_table
from src.features import feature_names_out
plt.rcParams.update({"figure.dpi": 150, "font.size": 10})
def unwrap(model):
    """Bóc lớp CalibratedClassifierCV để lấy pipeline gốc."""
    if hasattr(model, "calibrated_classifiers_"):
        est = model.calibrated_classifiers_[0].estimator
        # sklearn 1.9: mô hình prefit được bọc trong FrozenEstimator
        if est.__class__.__name__ == "FrozenEstimator":
            est = est.estimator
        return est
    return model
def error_by_group(X, y, proba, pred, col):
    rows = []
    for val, idx in X.groupby(col).groups.items():
        idx = list(idx)
        yy, pp, pr = y.iloc[idx], proba[idx], pred[idx]
        tp = int(((yy == 1) & (pr == 1)).sum())
        fn = int(((yy == 1) & (pr == 0)).sum())
        fp = int(((yy == 0) & (pr == 1)).sum())
        rows.append({"group": str(val), "n": len(idx), "n_yes": int(yy.sum()),
                     "recall": round(tp / (tp + fn), 4) if tp + fn else None,
                     "n_fp": fp, "n_fn": fn})
    return pd.DataFrame(rows).sort_values("recall")

In [ ]:
df = load_clean()
X_tr, X_va, X_te, y_tr, y_va, y_te = time_split_3way(df)
cal = joblib.load(MODELS_DIR / "model_calibrated.joblib")
base = joblib.load(MODELS_DIR / "model.joblib")
thr = json.loads((MODELS_DIR / "threshold.json").read_text(encoding="utf-8"))
cal_info = json.loads((REPORTS_DIR / "calibration_comparison.json").read_text(encoding="utf-8"))
final_name = cal_info["final_model"]
thr_cb = thr["threshold_cost_benefit"]
print(f"Mô hình cuối: {final_name} | calibrate: {thr['calibration_method']} | "
      f"ngưỡng chi phí-lợi ích: {thr_cb:.4f}")

## Đánh giá trên test ----------

In [ ]:
p_test = cal.predict_proba(X_te)[:, 1]
p_raw = base.predict_proba(X_te)[:, 1]
pred = (p_test >= thr_cb).astype(int)
m = compute_metrics(y_te, pred, p_test)
ci = bootstrap_ci(y_te, p_test, pred)
brier_raw = float(brier_score_loss(y_te, p_raw))
brier_cal = float(brier_score_loss(y_te, p_test))
print("Kết quả test theo thời gian (ngưỡng chi phí-lợi ích):",
      {k: round(v, 4) for k, v in m.items() if isinstance(v, float)})
print("Bootstrap CI 95%:", ci)
print(f"Brier trên test: gốc {brier_raw:.4f} -> đã calibrate {brier_cal:.4f}")

# Tham chiếu thêm: ngưỡng F1 trên validation (không phải quyết định mới —
# ngưỡng đã chọn trên validation, chỉ đánh giá thêm). Cần vì ngưỡng CB
# trên test gọi gần như toàn bộ khách nên không đọc được khả năng phân
# biệt của mô hình ở một điểm vận hành thực tế.
thr_f1 = thr["threshold_f1"]
pred_f1 = (p_test >= thr_f1).astype(int)
m_f1 = compute_metrics(y_te, pred_f1, p_test)
print("Tham chiếu — test ở ngưỡng F1:", {k: round(v, 4) for k, v in m_f1.items()
                                          if isinstance(v, float)})

summary = {"final_model": final_name, "selection_reason": cal_info["selection_reason"],
           "best_params": cal_info["best_params"],
           "calibration_method": thr["calibration_method"],
           "threshold_cost_benefit": round(thr_cb, 4),
           "threshold_f1_val": thr["threshold_f1"],
           "threshold_recall70_val": thr["threshold_recall70"],
           "test_metrics_cb_threshold": {k: (round(float(v), 4) if isinstance(v, float) else v)
                                         for k, v in m.items()},
           "test_metrics_f1_threshold": {k: (round(float(v), 4) if isinstance(v, float) else v)
                                         for k, v in m_f1.items()},
           "bootstrap_ci_95": ci,
           "brier_test_raw": round(brier_raw, 4),
           "brier_test_calibrated": round(brier_cal, 4)}
(REPORTS_DIR / "final_summary_time.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

## Hình ROC / PR / confusion trên test theo thời gian ----------

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
RocCurveDisplay.from_predictions(y_te, p_test, ax=axes[0])
axes[0].set_title(f"ROC curve - {final_name} (test theo thời gian)")
PrecisionRecallDisplay.from_predictions(y_te, p_test, ax=axes[1])
axes[1].set_title(f"PR curve - {final_name} (test theo thời gian)")
ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=["no", "yes"], ax=axes[2])
axes[2].set_title(f"Confusion matrix - {final_name} (test, ngưỡng CB={thr_cb:.3f})")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig17_final_time_curves.png", dpi=150)
plt.close(fig)
print("saved fig17_final_time_curves.png")

## Calibration curve trên test ----------

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for prob, lab in [(p_raw, "Mô hình gốc"), (p_test, f"Sau calibrate ({thr['calibration_method']})")]:
    fp, mp = calibration_curve(y_te, prob, n_bins=10, strategy="quantile")
    ax.plot(mp, fp, marker="o", label=lab)
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_xlabel("Xác suất dự đoán (trung bình bin)")
ax.set_ylabel("Tỷ lệ yes thực")
ax.set_title("Calibration curve (test theo thời gian)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig18_calibration_time.png", dpi=150)
plt.close(fig)
print("saved fig18_calibration_time.png")

## Phân tích lỗi theo nhóm + theo quý trên test (B4) ----------

In [ ]:
X_te_i = X_te.reset_index(drop=True)
y_te_i = y_te.reset_index(drop=True)
for col in ["month", "job", "poutcome"]:
    t = error_by_group(X_te_i, y_te_i, p_test, pred, col)
    t.to_csv(REPORTS_DIR / f"error_by_{col}_time.csv", index=False)
    print(f"Lỗi theo {col} (test): tệ nhất {t.iloc[0]['group']} "
          f"recall {t.iloc[0]['recall']}, tốt nhất {t.iloc[-1]['group']} "
          f"recall {t.iloc[-1]['recall']}")
# Chia test20 thành 4 quý theo thứ tự thời gian
q = np.array_split(np.arange(len(y_te_i)), 4)
qrows = []
for i, idx in enumerate(q):
    yy, pp, pr = y_te_i.iloc[idx], p_test[idx], pred[idx]
    mm = compute_metrics(yy, pr, pp)
    qrows.append({"quarter": f"Q{i + 1}", "n": len(idx),
                  "yes_rate": round(float(yy.mean()), 4),
                  "roc_auc": round(mm["roc_auc"], 4), "pr_auc": round(mm["pr_auc"], 4),
                  "recall": round(mm["recall"], 4), "precision": round(mm["precision"], 4)})
qdf = pd.DataFrame(qrows)
qdf.to_csv(REPORTS_DIR / "drift_by_quarter.csv", index=False)
print("Drift theo quý trên test:\n", qdf.to_string(index=False))

## Gain / lift trên test ----------

In [ ]:
res, dec = gain_lift_table(y_te, p_test)
dec.to_csv(REPORTS_DIR / "decile_table_time.csv")
top20 = float(res[res["called_pct"] <= 0.20]["captured"].iloc[-1])
print(f"Top 20% bắt được {top20 * 100:.1f}% số người đồng ý (test theo thời gian).")
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(res["called_pct"], res["captured"], label="Mô hình cuối")
ax.plot([0, 1], [0, 1], "--", label="Ngẫu nhiên")
ax.set_xlabel("% khách được gọi (xếp theo điểm)")
ax.set_ylabel("% khách đồng ý bắt được")
ax.set_title("Gain chart (test theo thời gian)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig22_gain_chart_time.png", dpi=150)
plt.close(fig)
print("saved fig22_gain_chart_time.png")

## B2: permutation importance + SHAP TRÊN VALIDATION ----------

In [ ]:
# Lưu ý: permutation_importance hoán vị các cột THÔ của X_va nên tên đặc
# trưng phải là cột gốc, không phải tên sau bước prep (số lượng khác nhau).
print("Permutation importance trên validation (n_repeats=3) ...")
perm = permutation_importance(cal, X_va, y_va, n_repeats=3, random_state=SEED,
                              scoring="average_precision", n_jobs=-1)
names = list(X_va.columns)
imp = pd.DataFrame({"feature": names,
                    "importance": perm.importances_mean,
                    "std": perm.importances_std}).sort_values(
                        "importance", ascending=False)
imp.to_csv(REPORTS_DIR / "permutation_importance_val.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 5))
top = imp.head(12).iloc[::-1]
ax.barh(top["feature"], top["importance"], xerr=top["std"])
ax.set_xlabel("Giảm PR-AUC khi hoán vị (validation)")
ax.set_title("Permutation importance — top 12 (validation)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig23_permutation_val.png", dpi=150)
plt.close(fig)
print("saved fig23_permutation_val.png | top 3:", imp.head(3)["feature"].tolist())

print("SHAP trên validation (mẫu 500 dòng) ...")
try:
    import shap
    rng = np.random.default_rng(SEED)
    sample_idx = rng.choice(len(X_va), size=min(500, len(X_va)), replace=False)
    Xs = X_va.iloc[sample_idx]
    pipe = unwrap(cal)
    clf = pipe.named_steps["clf"]
    x_prep = pipe.named_steps["prep"].transform(pipe.named_steps["fe"].transform(Xs))
    if clf.__class__.__name__ in ("RandomForestClassifier",
                                  "HistGradientBoostingClassifier",
                                  "LGBMClassifier", "DecisionTreeClassifier",
                                  "XGBClassifier"):
        vals = shap.TreeExplainer(clf).shap_values(x_prep)
        if isinstance(vals, list):
            vals = vals[1]
        elif getattr(vals, "ndim", 2) == 3:
            vals = vals[:, :, 1]
        mean_abs = np.abs(np.asarray(vals)).mean(axis=0)
    else:  # tuyến tính: dùng |hệ số| * độ lệch chuẩn đặc trưng
        mean_abs = np.abs(clf.coef_[0]) * np.asarray(x_prep).std(axis=0)
    shap_df = pd.DataFrame({"feature": list(feature_names_out(pipe)),
                            "mean_abs_shap": mean_abs}).sort_values(
        "mean_abs_shap", ascending=False)
    shap_df.to_csv(REPORTS_DIR / "shap_importance_val.csv", index=False)
    fig, ax = plt.subplots(figsize=(8, 5))
    top = shap_df.head(12).iloc[::-1]
    ax.barh(top["feature"], top["mean_abs_shap"])
    ax.set_xlabel("Mean |SHAP value| (validation)")
    ax.set_title("SHAP — |SHAP| trung bình, top 12 (validation)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig24_shap_val.png", dpi=150)
    plt.close(fig)
    print("saved fig24_shap_val.png | top 3:", shap_df.head(3)["feature"].tolist())
except Exception as e:
    print("SHAP bỏ qua:", type(e).__name__, e)

## Metadata mô hình ----------

In [ ]:
sha = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()[:16]
meta = {"model": final_name, "best_params": cal_info["best_params"],
        "calibration_method": thr["calibration_method"],
        "threshold_cost_benefit": round(thr_cb, 4),
        "threshold_f1_val": thr["threshold_f1"],
        "trained_at_utc": datetime.now(timezone.utc).isoformat(),
        "scikit_learn": sklearn.__version__,
        "data_file": RAW_PATH.name, "data_sha256_prefix": sha,
        "n_train": int(len(X_tr)), "n_val": int(len(X_va)), "n_test": int(len(X_te)),
        "test_metrics_cb_threshold": summary["test_metrics_cb_threshold"],
        "bootstrap_ci_95": ci,
        "brier_test_raw": round(brier_raw, 4),
        "brier_test_calibrated": round(brier_cal, 4),
        "top20_capture_test": round(top20, 4),
        "note": "Test theo thời gian (20% cuối). Đính chính 19/11/2026: test đã được đánh giá nhiều lần trong dự án (xem mục 4.2.8 của báo cáo); "
                "ngưỡng và calibrate chọn trên validation."}
(MODELS_DIR / "model_metadata.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã lưu models/model_metadata.json")

Giai đoạn 9b (v2): Thí nghiệm bổ sung trên thiết kế theo thời gian.

- Duration A/B (RQ4): TimeSeriesSplit CV trên train60 với mô hình cuối đã tuning.
- Ablation theo nhóm biến (RQ3): LogReg balanced, 7 tổ hợp, TimeSeriesSplit CV.
- B4 — Drift: vẽ PR-AUC/recall theo quý và recall theo tháng trên test theo
  thời gian; đề xuất chiến lược huấn luyện lại định kỳ (ghi trong báo cáo).
- B5 — Chi phí-lợi ích có khoảng bất định: đường lợi nhuận theo % khách được
  gọi cho 5 cặp (C, R), kèm dải tin cậy bootstrap 95%.

In [ ]:
import json
import os
import sys
from pathlib import Path
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
sys.path.insert(0, str(Path("..").resolve()))
from src.config import FIGURES_DIR, MODELS_DIR, REPORTS_DIR, SEED
from src.data import load_clean, time_split_3way
from src.evaluate import cv_report
from src.features import (GROUP_CAMPAIGN, GROUP_MACRO, GROUP_PERSONAL,
                          build_pipeline)
plt.rcParams.update({"figure.dpi": 150, "font.size": 10})
CLASSES = {"LogReg": LogisticRegression, "Tree": DecisionTreeClassifier,
           "KNN": KNeighborsClassifier, "HGB": HistGradientBoostingClassifier,
           "LightGBM": LGBMClassifier, "RF": RandomForestClassifier}
DEFAULT_KWARGS = {
    "LogReg": {"max_iter": 3000, "class_weight": "balanced", "random_state": SEED},
    "Tree": {"class_weight": "balanced", "random_state": SEED},
    "KNN": {"n_jobs": 1},
    "HGB": {"class_weight": "balanced", "random_state": SEED},
    "LightGBM": {"class_weight": "balanced", "verbosity": -1, "n_jobs": 1, "random_state": SEED},
    "RF": {"class_weight": "balanced_subsample", "n_jobs": 1, "random_state": SEED},
}
PAIRS = [(1.0, 15.0), (1.0, 10.0), (1.0, 25.0), (2.0, 15.0), (0.5, 15.0)]
def estimator_from_best(name, best_params):
    kwargs = {k.replace("clf__", ""): v for k, v in best_params[name].items()}
    return CLASSES[name](**{**DEFAULT_KWARGS[name], **kwargs})

In [ ]:
df = load_clean()
X_tr, X_va, X_te, y_tr, y_va, y_te = time_split_3way(df)
tscv = TimeSeriesSplit(n_splits=3)
best_params = json.loads((REPORTS_DIR / "best_params_time.json").read_text(encoding="utf-8"))
cal_info = json.loads((REPORTS_DIR / "calibration_comparison.json").read_text(encoding="utf-8"))
final_name = cal_info["final_model"]
out = {}

## Duration A/B trên thiết kế theo thời gian ----------

In [ ]:
est = estimator_from_best(final_name, best_params)
model_a = build_pipeline(type(est)(**est.get_params()))
model_b = build_pipeline(type(est)(**est.get_params()), include_duration=True)
ab = cv_report({"A_khong_duration": model_a, "B_co_duration": model_b},
               X_tr, y_tr, tscv)
ab.to_csv(REPORTS_DIR / "duration_ab_time_cv.csv")
out["duration_ab_time_cv"] = ab[["roc_auc_cv", "pr_auc_cv"]].round(4).to_dict("index")
print("Duration A/B (TimeSeriesSplit CV, train60):", out["duration_ab_time_cv"])

## Ablation theo nhóm biến (TimeSeriesSplit CV) ----------

In [ ]:
groups = {"Personal": GROUP_PERSONAL, "Campaign": GROUP_CAMPAIGN, "Macro": GROUP_MACRO}
combos = {"Personal": ["Personal"], "Campaign": ["Campaign"], "Macro": ["Macro"],
          "Personal+Campaign": ["Personal", "Campaign"],
          "Campaign+Macro": ["Campaign", "Macro"],
          "Personal+Macro": ["Personal", "Macro"],
          "All": ["Personal", "Campaign", "Macro"]}

def merge(names):
    num = sum([groups[n]["num"] for n in names], [])
    cat = sum([groups[n]["cat"] for n in names], [])
    return num, cat

abl = {}
for label, names in combos.items():
    num, cat = merge(names)
    abl[label] = build_pipeline(
        LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED),
        num=num, cat=cat)
abl_res = cv_report(abl, X_tr, y_tr, tscv)
abl_res.to_csv(REPORTS_DIR / "ablation_time_cv.csv")
out["ablation_time_cv"] = abl_res[["roc_auc_cv", "pr_auc_cv", "f1_cv",
                                    "recall_cv"]].round(4).to_dict("index")
print("Ablation (TimeSeriesSplit CV):\n", abl_res[["roc_auc_cv", "pr_auc_cv"]].to_string())

## B4: drift theo quý/tháng trên test ----------

In [ ]:
cal = joblib.load(MODELS_DIR / "model_calibrated.joblib")
p_test = cal.predict_proba(X_te)[:, 1]
thr = json.loads((MODELS_DIR / "threshold.json").read_text(encoding="utf-8"))
pred = (p_test >= thr["threshold_cost_benefit"]).astype(int)
qdf = pd.read_csv(REPORTS_DIR / "drift_by_quarter.csv")
fig, ax1 = plt.subplots(figsize=(7, 4.5))
ax1.plot(qdf["quarter"], qdf["yes_rate"], marker="o", label="Tỷ lệ yes thực")
ax1.plot(qdf["quarter"], qdf["recall"], marker="s", label="Recall")
ax1.plot(qdf["quarter"], qdf["pr_auc"], marker="^", label="PR-AUC")
ax1.set_ylim(0, 1.05)
ax1.set_xlabel("Quý trong tập test theo thời gian (Q1 sớm nhất)")
ax1.set_ylabel("Tỷ lệ / điểm số")
ax1.set_title("Độ trôi trên test theo thời gian: yes-rate, recall, PR-AUC theo quý")
ax1.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig19_drift_quarter.png", dpi=150)
plt.close(fig)
print("saved fig19_drift_quarter.png")

err_month = pd.read_csv(REPORTS_DIR / "error_by_month_time.csv")
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(err_month["group"], err_month["recall"].fillna(0))
ax.set_ylim(0, 1.05)
ax.set_xlabel("Tháng (test theo thời gian)")
ax.set_ylabel("Recall lớp yes")
ax.set_title("Recall theo tháng trên test theo thời gian")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig20_drift_month.png", dpi=150)
plt.close(fig)
print("saved fig20_drift_month.png")

## B5: đường lợi nhuận theo % gọi, kèm bootstrap ----------

In [ ]:
y = np.asarray(y_te)
order = np.argsort(-p_test)
y_sorted = y[order]
grid = np.linspace(0.05, 1.0, 20)  # % khách được gọi
rng = np.random.default_rng(SEED)
curves = {}
for C, R in PAIRS:
    profits = []
    for _ in range(200):
        idx = rng.integers(0, len(y), len(y))
        ys = y_sorted[idx]
        cum = np.cumsum(ys)
        k = np.maximum((grid * len(y)).astype(int), 1)
        profits.append(R * cum[k - 1] - C * k)
    profits = np.array(profits)
    curves[f"C={C},R={R}"] = {
        "called_pct": grid.round(3).tolist(),
        "profit_mean": profits.mean(axis=0).round(1).tolist(),
        "profit_lo": np.percentile(profits, 2.5, axis=0).round(1).tolist(),
        "profit_hi": np.percentile(profits, 97.5, axis=0).round(1).tolist(),
    }
(REPORTS_DIR / "cost_benefit_curves_time.json").write_text(
    json.dumps(curves, ensure_ascii=False, indent=2), encoding="utf-8")
fig, ax = plt.subplots(figsize=(8, 5))
for label, c in curves.items():
    x = np.array(c["called_pct"]) * 100
    ax.plot(x, c["profit_mean"], marker="o", ms=3, label=label)
    ax.fill_between(x, c["profit_lo"], c["profit_hi"], alpha=0.15)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("% khách được gọi (xếp theo điểm)")
ax.set_ylabel("Lợi nhuận (đơn vị tiền tệ giả định)")
ax.set_title("Lợi nhuận theo % khách được gọi — dải mờ là CI 95% (bootstrap)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig21_profit_curves.png", dpi=150)
plt.close(fig)
print("saved fig21_profit_curves.png")
# Điểm tối ưu cho từng cặp (trên test, chỉ để tham chiếu)
opt = {}
for label, c in curves.items():
    i = int(np.argmax(c["profit_mean"]))
    opt[label] = {"best_called_pct": round(float(c["called_pct"][i]) * 100, 1),
                  "profit_mean": c["profit_mean"][i]}
out["cost_benefit_optimum_test"] = opt
print("Điểm tối ưu lợi nhuận trên test:", opt)

(REPORTS_DIR / "extra_experiments_time.json").write_text(
    json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã lưu reports/extra_experiments_time.json")